# Figure 1 — R² performance: Original vs 90% Filtered datasets

Loads `Results_{target}.csv` from `Original_DS` and `Clean_90_DS`, builds a
summary table, and produces publication-ready bar charts.

**Original**: trained on original data. Train R² from `Original_DS` (unaffected
by the criterion-1/HR-interval fix below - the unfiltered training set never
had that filter applied). Test R² from `Sweep_Threshold_ML.py`'s thres_90
"unfiltered" point estimate (the original-trained model evaluated on the
*corrected* 90%-filtered test set) - NOT from `Original_DS`'s own `Test_90`
row, which was computed against the test set before the fix below.

**90 Threshold**: trained on 90%-quality-filtered data. Both Train and Test R²
come from `Clean_90_DS`, freshly regenerated by rerunning `Experiment_eval.py`
against the corrected `Clean_Features_VitalDB_Train_Subset_90.h5` /
`Clean_Features_VitalDB_CalFree_Test_Subset_90.h5`.

**Why this rebuild was needed**: the fiducial quality checker's criterion 1
(heart-rate plausibility check) used the wrong interval (`bmin=50, bmax=180`)
when the datasets/models behind the original version of this figure were
produced. It's since been corrected to `bmin=40, bmax=200`
(`Scripts/fiducial_quality_filtering/run_pipeline.py`), and
`Clean_Features_*_90.h5` was rebuilt from the corrected dropped-signal JSONs.
This notebook re-sources every number affected by that interval mismatch.

In [ ]:
import sys
from pathlib import Path

# figures/ → fig_style.py  |  bp_lgbm/ → local_paths.py
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path("../").resolve()))

from fig_style import (
    DPI, W_FULL, W_SINGLE, ASPECT, FONT_FAMILY,
    MIN_PX_FULL, MIN_PX_SINGLE,
    apply_base_style, save_fig,
)
from local_paths import PERFORMANCE_RESULTS_PAPER, FIGURES_PAPER, THRESHOLD_SWEEP_ML_RESULTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FIG_OUT = FIGURES_PAPER
FIG_OUT.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {FIG_OUT.resolve()}")
print(f"Full-page : {W_FULL:.2f} × {W_FULL*ASPECT:.2f} in  →  {round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px @ {DPI} dpi")
print(f"Single-col: {W_SINGLE:.2f} × {W_SINGLE*ASPECT:.2f} in  →  {round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px @ {DPI} dpi")

## 1 · Load results

In [ ]:
TARGETS = ["SBP", "DBP", "MAP"]

def load_experiment(folder: str, targets=TARGETS) -> dict[str, pd.DataFrame]:
    """Load Results_{target}.csv for each target from a subfolder of PERFORMANCE_RESULTS_PAPER."""
    base = PERFORMANCE_RESULTS_PAPER / folder
    dfs = {}
    for t in targets:
        p = base / f"Results_{t}.csv"
        # Handle both comma and semicolon separators (legacy files used ';')
        raw = p.read_text(encoding="utf-8")
        sep = ";" if raw.count(";") > raw.count(",") else ","
        df = pd.read_csv(p, sep=sep)
        df.columns = df.columns.str.strip()
        df["Data_Subset"] = df["Data_Subset"].str.strip()
        df.set_index("Data_Subset", inplace=True)
        dfs[t] = df
        print(f"  {folder}/{t}: rows = {list(df.index)}")
    return dfs

print("Loading Original_DS...")
orig = load_experiment("Original_DS")

print("Loading Clean_90_DS (freshly rebuilt against the corrected criterion-1 interval)...")
c90  = load_experiment("Clean_90_DS")

# The Original-trained model's TEST R2 has to be re-sourced too: Original_DS's
# own Test_90 row was evaluated against the test set BEFORE the criterion-1
# fix. Sweep_Threshold_ML.py already re-evaluated that same original/unfiltered
# model against the corrected 90%-filtered test set (thres_90's "unfiltered"
# point estimate) - use that instead. R2 there is a raw fraction, same as the
# Results_{target}.csv files above, so no unit conversion is needed.
print("Loading Sweep_Threshold_ML.py thres_90 point estimates (corrected test set)...")
sweep_r2_thres90 = pd.read_csv(THRESHOLD_SWEEP_ML_RESULTS / "thres_90" / "PointEstimates_R2.csv").set_index("target")
print(sweep_r2_thres90)


## 2 · Build summary table

For **Original**: Train row → `Original_DS` Train R², Test → sweep thres_90 "unfiltered" R² (corrected test set)
For **90 Threshold**: Train row → `Clean_90_DS` Train R², Test_90 row → `Clean_90_DS` Test_90 R² (both freshly rebuilt)

In [ ]:
rows = []
for target in TARGETS:
    rows.append({"Target": target, "Split": "Train", "Dataset": "Original",
                 "R2": orig[target].loc["Train", "R2"]})
    rows.append({"Target": target, "Split": "Test", "Dataset": "Original",
                 "R2": sweep_r2_thres90.loc[target, "unfiltered"]})
    rows.append({"Target": target, "Split": "Train", "Dataset": "90 Threshold",
                 "R2": c90[target].loc["Train", "R2"]})
    rows.append({"Target": target, "Split": "Test", "Dataset": "90 Threshold",
                 "R2": c90[target].loc["Test_90", "R2"]})

summary = pd.DataFrame(rows)
summary["R2_pct"] = summary["R2"] * 100

# Pivot for easy reading
pivot = summary.pivot_table(index=["Target", "Split"], columns="Dataset", values="R2_pct")
pivot = pivot[["Original", "90 Threshold"]]  # enforce column order
print(pivot.round(2).to_string())
pivot.round(4).to_csv(FIG_OUT / "Table_R2_OriginalVs90.csv")
print(f"Table saved -> {FIG_OUT / 'Table_R2_OriginalVs90.csv'}")


## 3 · Plot — grouped bar chart

Layout mirrors the reference figure: 6 x-positions (Train/Test × SBP/DBP/MAP), two bars each.

In [ ]:
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

# ── Figure-specific palette ──
COLOR_ORIG  = "#C0C0C0"   # light gray  → Original dataset
COLOR_90    = "#1F4E79"   # dark navy   → 90 Threshold dataset
HATCH_TEST  = "///"       # Test split
BAR_WIDTH   = 0.35
LABEL_NUDGE = 0.09        # data-unit nudge: orig left, 90T right

# X ordering: Train SBP, Test SBP, Train DBP, Test DBP, Train MAP, Test MAP
order = [(t, s) for t in TARGETS for s in ["Train", "Test"]]
x     = np.arange(len(order))

orig_vals, c90_vals = [], []
for target, split in order:
    row = summary.query("Target == @target and Split == @split")
    orig_vals.append(row.query("Dataset == 'Original'")["R2_pct"].values[0])
    c90_vals.append(row.query("Dataset == '90 Threshold'")["R2_pct"].values[0])


def make_r2_barchart(width_in: float):
    _prev_hatch_lw = mpl.rcParams['hatch.linewidth']
    mpl.rcParams['hatch.linewidth'] = 2.0

    fig, ax = plt.subplots(figsize=(width_in, width_in * ASPECT),
                           dpi=DPI, layout="constrained")

    # Pass 1: solid fill + uniform black outline on ALL bars (same zorder)
    # Pass 2: transparent white-hatch overlay for Test bars only (no extra border)
    # Using same zorder=3 for both passes avoids edge-clipping artefacts
    all_bars = []  # (rect, "orig" | "c90")
    for i, (target, split) in enumerate(order):
        is_test = split == "Test"

        bc1 = ax.bar(i - BAR_WIDTH/2, orig_vals[i], BAR_WIDTH,
                     color=COLOR_ORIG, facecolor=COLOR_ORIG,
                     edgecolor="black", linewidth=0.8, zorder=3)
        bc2 = ax.bar(i + BAR_WIDTH/2, c90_vals[i], BAR_WIDTH,
                     color=COLOR_90, facecolor=COLOR_90,
                     edgecolor="black", linewidth=0.8, zorder=3)

        if is_test:
            # Overlay: transparent fill, white hatch, zero-width border (same zorder)
            ax.bar(i - BAR_WIDTH/2, orig_vals[i], BAR_WIDTH,
                   facecolor="none", hatch=HATCH_TEST,
                   edgecolor="white", linewidth=0, zorder=3)
            ax.bar(i + BAR_WIDTH/2, c90_vals[i], BAR_WIDTH,
                   facecolor="none", hatch=HATCH_TEST,
                   edgecolor="white", linewidth=0, zorder=3)

        all_bars.extend([(bc1[0], "orig"), (bc2[0], "c90")])

    # Data labels — orig nudged left, 90T nudged right
    label_fs = 7 if width_in < 5 else 10
    for bar, which in all_bars:
        h = bar.get_height()
        cx = bar.get_x() + bar.get_width() / 2
        dx = -LABEL_NUDGE if which == "orig" else +LABEL_NUDGE
        ax.text(cx + dx, h + 0.4, f"{h:.2f}%", ha="center", va="bottom",
                fontsize=label_fs, fontfamily=FONT_FAMILY, color="#333333")

    # Y-axis — capped at 70%
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(10))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
    ax.set_ylabel("R²", fontsize=10 if width_in < 5 else 13, fontfamily=FONT_FAMILY)

    # X-axis tick labels
    tick_fs = 8 if width_in < 5 else 11
    ax.set_xticks(x)
    ax.set_xticklabels([s for (_, s) in order], fontsize=tick_fs, fontfamily=FONT_FAMILY)

    # Target names just below tick labels
    target_fs = 9 if width_in < 5 else 12
    for cx, tgt in zip([0.5, 2.5, 4.5], TARGETS):
        ax.text(cx, -0.14, tgt, ha="center", va="top",
                fontsize=target_fs, fontfamily=FONT_FAMILY,
                fontweight="bold", transform=ax.get_xaxis_transform())

    # Group dividers
    for xd in [1.5, 3.5]:
        ax.axvline(xd, color="#CCCCCC", linewidth=0.8, linestyle="--", zorder=0)

    apply_base_style(ax, grid_axis="y")
    ax.tick_params(axis="both", labelsize=tick_fs)

    # Title
    title_fs = 8 if width_in < 5 else 12
    ax.set_title("R² performance across targets in original vs\nfiltered (90% threshold) datasets",
                 fontsize=title_fs, fontfamily=FONT_FAMILY, pad=6)

    # Legend — upper right, inside axes
    # Test entry uses HandlerTuple(ndivide=1) to overlay two patches in one key:
    # layer 1: gray fill + black outline; layer 2: transparent + white hatch
    leg_fs = 7 if width_in < 5 else 10
    test_handle = (
        Patch(facecolor="#D0D0D0", edgecolor="black", linewidth=0.8),
        Patch(facecolor="none",    edgecolor="white", hatch=HATCH_TEST, linewidth=0),
    )
    legend_elements = [
        Patch(facecolor=COLOR_ORIG, edgecolor="black", linewidth=0.8, label="Original"),
        Patch(facecolor=COLOR_90,   edgecolor="black", linewidth=0.8, label="90 Threshold"),
        Patch(facecolor="#D0D0D0",  edgecolor="black", linewidth=0.8, label="Train"),
        test_handle,
    ]
    ax.legend(
        handles=legend_elements,
        labels=["Original", "90 Threshold", "Train", "Test"],
        handler_map={tuple: HandlerTuple(ndivide=1, pad=0)},
        loc="upper right", ncol=1, fontsize=leg_fs,
        frameon=True, framealpha=0.9, edgecolor="#CCCCCC",
    )

    mpl.rcParams['hatch.linewidth'] = _prev_hatch_lw
    return fig


fig_full   = make_r2_barchart(W_FULL)
fig_single = make_r2_barchart(W_SINGLE)

save_fig(fig_full,   "Fig1_R2_OriginalVs90_full",   FIG_OUT)
save_fig(fig_single, "Fig1_R2_OriginalVs90_single", FIG_OUT)

print(f"Full page : {round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px")
print(f"Single col: {round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px")

plt.show()

## 4 · Verify pixel counts

Confirm PNG output meets the minimum pixel requirements for the journal.

In [ ]:
from PIL import Image

for fname, req_w in [
    ("Fig1_R2_OriginalVs90_full.png",   MIN_PX_FULL),
    ("Fig1_R2_OriginalVs90_single.png", MIN_PX_SINGLE),
]:
    with Image.open(FIG_OUT / fname) as im:
        w, h = im.size
    ok = "✅" if w >= req_w else "❌"
    print(f"{ok} {fname}: {w} × {h} px  (min required: {req_w} px wide)")